# Chapter 1.7: Quantization Primer

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** floating-point formats: FP32, FP16, BF16, FP8
2. **Distinguish** PTQ (Post-Training Quantization) from QAT (Quantization-Aware Training)
3. **Implement** symmetric and asymmetric quantization from scratch
4. **Compare** per-tensor, per-channel, and per-group quantization
5. **Explain** GPTQ and AWQ quantization methods
6. **Analyze** quality vs speed trade-offs

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.7_quantization.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.7_quantization.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import struct

torch.manual_seed(42)
np.random.seed(42)
print("Ready to explore quantization!")

---

## 1. Floating-Point Number Formats

### 1.1 IEEE 754 Floating-Point

A floating-point number is represented as:
$$(-1)^{\text{sign}} \times 2^{\text{exponent} - \text{bias}} \times (1 + \text{mantissa})$$

| Format | Total bits | Sign | Exponent | Mantissa | Range | Precision |
|--------|-----------|------|----------|----------|-------|-----------|
| FP32   | 32 | 1 | 8 | 23 | ~3.4e38 | ~7 digits |
| FP16   | 16 | 1 | 5 | 10 | ~6.5e4 | ~3.3 digits |
| BF16   | 16 | 1 | 8 | 7 | ~3.4e38 | ~2.4 digits |
| FP8 E4M3 | 8 | 1 | 4 | 3 | ~448 | ~1.4 digits |
| FP8 E5M2 | 8 | 1 | 5 | 2 | ~57344 | ~1.1 digits |

In [ ]:
## Figure: Floating-Point Format Bit Layouts
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 35)
ax.set_ylim(0, 18)
ax.axis('off')
ax.set_title('Number Format Comparison: Bit Layouts', fontsize=16, fontweight='bold')

SIGN_COLOR = '#8E44AD'
EXP_COLOR = '#4A90D9'
MANT_COLOR = '#27AE60'

formats = [
    ('FP32', 1, 8, 23, 32, '~7 decimal digits'),
    ('FP16', 1, 5, 10, 16, '~3.3 decimal digits'),
    ('BF16', 1, 8, 7, 16, '~2.4 digits, FP32 range'),
    ('FP8 (E4M3)', 1, 4, 3, 8, '~1.4 digits'),
    ('FP8 (E5M2)', 1, 5, 2, 8, '~1.1 digits, wider range'),
    ('INT8', 1, 0, 7, 8, '256 levels'),
    ('INT4', 1, 0, 3, 4, '16 levels'),
]

y_start = 16.5
y_step = 2.2
scale = 0.55  # width per bit

for i, (name, sign_bits, exp_bits, mant_bits, total, desc) in enumerate(formats):
    y = y_start - i * y_step
    x_start = 8
    
    # Format name
    ax.text(1, y + 0.3, name, fontsize=12, fontweight='bold', va='center')
    ax.text(1, y - 0.3, f'{total} bits', fontsize=9, va='center', color='gray')
    
    x = x_start
    
    # Sign bit
    if sign_bits > 0:
        w = sign_bits * scale
        ax.add_patch(patches.Rectangle((x, y - 0.4), w, 0.8, facecolor=SIGN_COLOR, 
                                        alpha=0.8, edgecolor='black', linewidth=1))
        ax.text(x + w/2, y, 'S', ha='center', va='center', fontsize=7, fontweight='bold', color='white')
        x += w
    
    # Exponent bits
    if exp_bits > 0:
        w = exp_bits * scale
        ax.add_patch(patches.Rectangle((x, y - 0.4), w, 0.8, facecolor=EXP_COLOR,
                                        alpha=0.8, edgecolor='black', linewidth=1))
        ax.text(x + w/2, y, f'Exp ({exp_bits})', ha='center', va='center', fontsize=7, 
                fontweight='bold', color='white')
        x += w
    
    # Mantissa bits
    mant_label = f'Mantissa ({mant_bits})' if exp_bits > 0 else f'Integer ({mant_bits})'
    if mant_bits > 0:
        w = mant_bits * scale
        ax.add_patch(patches.Rectangle((x, y - 0.4), w, 0.8, facecolor=MANT_COLOR,
                                        alpha=0.8, edgecolor='black', linewidth=1))
        ax.text(x + w/2, y, mant_label, ha='center', va='center', fontsize=7,
                fontweight='bold', color='white')
        x += w
    
    # Description
    ax.text(x + 0.5, y, desc, fontsize=9, va='center', style='italic', color='#2C3E50')
    
    # Model size annotation (7B model)
    model_size = 7e9 * total / 8 / 1e9
    ax.text(x + 0.5, y - 0.35, f'7B model: {model_size:.1f} GB', fontsize=8, va='center', color='gray')

# Legend
legend_y = 0.5
for color, label in [(SIGN_COLOR, 'Sign'), (EXP_COLOR, 'Exponent (range)'), (MANT_COLOR, 'Mantissa (precision)')]:
    ax.add_patch(patches.Rectangle((10, legend_y), 1.5, 0.5, facecolor=color, alpha=0.8, edgecolor='black'))
    ax.text(12, legend_y + 0.25, label, fontsize=10, va='center', fontweight='bold')
    legend_y += 0.8

plt.tight_layout()
plt.show()
print("BF16 has same range as FP32 (8-bit exponent) but less precision (7-bit mantissa).")
print("This makes BF16 ideal for training -- it handles the same value range as FP32.")

In [ ]:
def visualize_float_bits(value, dtype_name='FP32'):
    """Visualize the bit representation of a floating-point number."""
    if dtype_name == 'FP32':
        packed = struct.pack('!f', value)
        bits = ''.join(f'{b:08b}' for b in packed)
        sign, exp, mantissa = bits[0], bits[1:9], bits[9:]
        exp_bits, man_bits = 8, 23
    elif dtype_name == 'FP16':
        t = torch.tensor(value, dtype=torch.float16)
        packed = struct.pack('!H', t.view(torch.int16).item() & 0xFFFF)
        bits = ''.join(f'{b:08b}' for b in packed)
        sign, exp, mantissa = bits[0], bits[1:6], bits[6:]
        exp_bits, man_bits = 5, 10
    elif dtype_name == 'BF16':
        t = torch.tensor(value, dtype=torch.bfloat16)
        packed = struct.pack('!H', t.view(torch.int16).item() & 0xFFFF)
        bits = ''.join(f'{b:08b}' for b in packed)
        sign, exp, mantissa = bits[0], bits[1:9], bits[9:]
        exp_bits, man_bits = 8, 7
    
    return {
        'value': value,
        'dtype': dtype_name,
        'sign': sign,
        'exponent': exp,
        'mantissa': mantissa,
        'total_bits': 1 + len(exp) + len(mantissa)
    }


# Show representations of pi
value = 3.14159
print(f"Representing {value} in different formats:")
print("=" * 60)

for dtype in ['FP32', 'FP16', 'BF16']:
    info = visualize_float_bits(value, dtype)
    print(f"\n{dtype} ({info['total_bits']} bits):")
    print(f"  Sign:     {info['sign']}")
    print(f"  Exponent: {info['exponent']}")
    print(f"  Mantissa: {info['mantissa']}")
    
    # Reconstruct value
    if dtype == 'FP32':
        actual = torch.tensor(value, dtype=torch.float32).item()
    elif dtype == 'FP16':
        actual = torch.tensor(value, dtype=torch.float16).item()
    elif dtype == 'BF16':
        actual = torch.tensor(value, dtype=torch.bfloat16).item()
    print(f"  Stored as: {actual:.10f} (error: {abs(value - actual):.10f})")

In [ ]:
# Visualize the representable number distribution for different formats

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# FP16 vs BF16 precision comparison
x_fp32 = torch.linspace(-10, 10, 10000, dtype=torch.float32)

# FP16 quantization error
x_fp16 = x_fp32.to(torch.float16).to(torch.float32)
fp16_error = (x_fp32 - x_fp16).abs()

# BF16 quantization error
x_bf16 = x_fp32.to(torch.bfloat16).to(torch.float32)
bf16_error = (x_fp32 - x_bf16).abs()

axes[0, 0].plot(x_fp32.numpy(), fp16_error.numpy(), 'b-', alpha=0.5, label='FP16 error')
axes[0, 0].plot(x_fp32.numpy(), bf16_error.numpy(), 'r-', alpha=0.5, label='BF16 error')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Absolute Error')
axes[0, 0].set_title('Representation Error: FP16 vs BF16')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Dynamic range comparison
ranges = {
    'FP32': (1.18e-38, 3.4e38),
    'FP16': (6.1e-5, 6.5e4),
    'BF16': (1.18e-38, 3.4e38),
    'INT8': (-128, 127),
    'INT4': (-8, 7),
}

names = list(ranges.keys())
log_ranges = [np.log10(abs(r[1])) + np.log10(max(abs(r[0]), 1e-40)) for r in ranges.values()]

axes[0, 1].barh(names, [np.log10(abs(r[1])) for r in ranges.values()], 
                color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'], alpha=0.8)
axes[0, 1].set_xlabel('log10(max value)')
axes[0, 1].set_title('Dynamic Range Comparison')
axes[0, 1].grid(True, alpha=0.3)

# Memory savings
formats = ['FP32', 'FP16/BF16', 'INT8', 'INT4', 'INT2']
bits = [32, 16, 8, 4, 2]
model_7b_gb = [7e9 * b / 8 / 1e9 for b in bits]

axes[1, 0].bar(formats, model_7b_gb, color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'], alpha=0.8)
axes[1, 0].set_ylabel('Model Size (GB)')
axes[1, 0].set_title('7B Parameter Model Size by Format')
for i, (fmt, sz) in enumerate(zip(formats, model_7b_gb)):
    axes[1, 0].text(i, sz + 0.3, f'{sz:.1f} GB', ha='center', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Throughput improvement (theoretical, memory-bound)
speedups = [bits[1] / b for b in bits]  # relative to FP16
axes[1, 1].bar(formats, speedups, color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'], alpha=0.8)
axes[1, 1].set_ylabel('Theoretical Speedup (vs FP16)')
axes[1, 1].set_title('Decode Speedup (Memory-Bound Limit)')
for i, s in enumerate(speedups):
    axes[1, 1].text(i, s + 0.05, f'{s:.1f}x', ha='center', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
## Figure: Quantization Error Visualization
import matplotlib.pyplot as plt
import numpy as np
import torch

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

torch.manual_seed(42)
weights = torch.randn(1000) * 0.5  # typical weight distribution

for ax, n_bits, title_suffix in zip(axes, [8, 4, 2], ['INT8', 'INT4', 'INT2']):
    qmax = 2 ** (n_bits - 1) - 1
    scale = weights.abs().max() / qmax
    w_q = torch.clamp(torch.round(weights / scale), -qmax, qmax)
    w_hat = w_q * scale
    
    ax.scatter(weights.numpy(), w_hat.numpy(), alpha=0.3, s=8, c='#4A90D9', label='Quantized')
    
    # Perfect line
    lims = [weights.min().item() - 0.1, weights.max().item() + 0.1]
    ax.plot(lims, lims, 'r--', linewidth=1.5, alpha=0.7, label='Perfect (no error)')
    
    mae = (weights - w_hat).abs().mean().item()
    ax.set_xlabel('Original Weight', fontsize=11)
    ax.set_ylabel('Quantized Weight', fontsize=11)
    ax.set_title(f'{title_suffix} Quantization\nMAE = {mae:.4f}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.suptitle('Original vs Quantized Weights: Quantization Error', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("INT8: Nearly perfect reconstruction. INT4: Visible stepping. INT2: Severe information loss.")

---

## 2. Quantization Fundamentals

### 2.1 What is Quantization?

Quantization maps floating-point values to a smaller set of discrete values (typically integers).

### 2.2 Symmetric Quantization

$$x_q = \text{round}\left(\frac{x}{s}\right), \quad s = \frac{\max(|x|)}{2^{b-1} - 1}$$

$$\hat{x} = x_q \times s \quad (\text{dequantization})$$

### 2.3 Asymmetric Quantization

$$x_q = \text{round}\left(\frac{x - z}{s}\right), \quad s = \frac{\max(x) - \min(x)}{2^b - 1}, \quad z = \min(x)$$

$$\hat{x} = x_q \times s + z$$

In [ ]:
def symmetric_quantize(x: torch.Tensor, n_bits: int = 8) -> tuple:
    """
    Symmetric quantization: map float values to [-2^(b-1), 2^(b-1)-1]
    Zero maps to zero.
    """
    qmin = -(2 ** (n_bits - 1))
    qmax = 2 ** (n_bits - 1) - 1
    
    # Compute scale
    abs_max = x.abs().max()
    scale = abs_max / qmax
    
    # Quantize
    x_q = torch.clamp(torch.round(x / scale), qmin, qmax).to(torch.int8 if n_bits == 8 else torch.int32)
    
    return x_q, scale


def symmetric_dequantize(x_q: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize: int -> float"""
    return x_q.float() * scale


def asymmetric_quantize(x: torch.Tensor, n_bits: int = 8) -> tuple:
    """
    Asymmetric quantization: map float values to [0, 2^b - 1]
    Uses a zero-point offset.
    """
    qmin = 0
    qmax = 2 ** n_bits - 1
    
    x_min = x.min()
    x_max = x.max()
    
    scale = (x_max - x_min) / (qmax - qmin)
    zero_point = qmin - torch.round(x_min / scale)
    zero_point = torch.clamp(zero_point, qmin, qmax)
    
    x_q = torch.clamp(torch.round(x / scale + zero_point), qmin, qmax).to(torch.uint8 if n_bits == 8 else torch.int32)
    
    return x_q, scale, zero_point


def asymmetric_dequantize(x_q: torch.Tensor, scale: float, zero_point: float) -> torch.Tensor:
    return (x_q.float() - zero_point) * scale


# Demo
x = torch.randn(1000) * 2 + 0.5  # non-zero-centered distribution

# Symmetric quantization
x_q_sym, scale_sym = symmetric_quantize(x, n_bits=8)
x_hat_sym = symmetric_dequantize(x_q_sym, scale_sym)
error_sym = (x - x_hat_sym).abs().mean()

# Asymmetric quantization
x_q_asym, scale_asym, zp_asym = asymmetric_quantize(x, n_bits=8)
x_hat_asym = asymmetric_dequantize(x_q_asym, scale_asym, zp_asym)
error_asym = (x - x_hat_asym).abs().mean()

print(f"Original:  mean={x.mean():.4f}, std={x.std():.4f}")
print(f"\nSymmetric INT8:")
print(f"  Scale: {scale_sym:.6f}")
print(f"  MAE: {error_sym:.6f}")
print(f"\nAsymmetric INT8:")
print(f"  Scale: {scale_asym:.6f}, Zero Point: {zp_asym:.0f}")
print(f"  MAE: {error_asym:.6f}")
print(f"\nAsymmetric is {error_sym/error_asym:.2f}x more accurate for this distribution")

In [ ]:
# Visualize quantization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original distribution
axes[0, 0].hist(x.numpy(), bins=100, alpha=0.7, color='blue', label='Original (FP32)')
axes[0, 0].set_title('Original Distribution', fontsize=13)
axes[0, 0].legend()

# Symmetric quantized
axes[0, 1].hist(x_hat_sym.numpy(), bins=100, alpha=0.7, color='green', label='Symmetric INT8')
axes[0, 1].hist(x.numpy(), bins=100, alpha=0.3, color='blue', label='Original')
axes[0, 1].set_title(f'Symmetric Quantization (MAE={error_sym:.4f})', fontsize=13)
axes[0, 1].legend()

# Asymmetric quantized
axes[1, 0].hist(x_hat_asym.numpy(), bins=100, alpha=0.7, color='red', label='Asymmetric INT8')
axes[1, 0].hist(x.numpy(), bins=100, alpha=0.3, color='blue', label='Original')
axes[1, 0].set_title(f'Asymmetric Quantization (MAE={error_asym:.4f})', fontsize=13)
axes[1, 0].legend()

# Error comparison across bit widths
bit_widths = [2, 3, 4, 5, 6, 7, 8]
errors_sym = []
errors_asym = []
for bits in bit_widths:
    xq, s = symmetric_quantize(x, bits)
    errors_sym.append((x - symmetric_dequantize(xq, s)).abs().mean().item())
    
    xq, s, zp = asymmetric_quantize(x, bits)
    errors_asym.append((x - asymmetric_dequantize(xq, s, zp)).abs().mean().item())

axes[1, 1].plot(bit_widths, errors_sym, 'b-o', linewidth=2, label='Symmetric')
axes[1, 1].plot(bit_widths, errors_asym, 'r-o', linewidth=2, label='Asymmetric')
axes[1, 1].set_xlabel('Bit Width', fontsize=12)
axes[1, 1].set_ylabel('Mean Absolute Error', fontsize=12)
axes[1, 1].set_title('Quantization Error vs Bit Width', fontsize=13)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

---

## 3. Granularity: Per-Tensor vs Per-Channel vs Per-Group

The granularity determines how many elements share the same scale/zero-point:

- **Per-tensor**: One scale for the entire weight matrix
- **Per-channel**: One scale per output channel (row)
- **Per-group**: One scale per group of `g` elements within a channel

In [ ]:
def per_tensor_quantize(weight: torch.Tensor, n_bits: int = 8) -> tuple:
    """One scale for the entire tensor."""
    x_q, scale = symmetric_quantize(weight.flatten(), n_bits)
    return x_q.reshape(weight.shape), scale

def per_channel_quantize(weight: torch.Tensor, n_bits: int = 8) -> tuple:
    """One scale per row (output channel)."""
    scales = []
    x_q_rows = []
    for i in range(weight.shape[0]):
        row_q, scale = symmetric_quantize(weight[i], n_bits)
        x_q_rows.append(row_q)
        scales.append(scale)
    return torch.stack(x_q_rows), torch.tensor(scales)

def per_group_quantize(weight: torch.Tensor, n_bits: int = 8, group_size: int = 128) -> tuple:
    """One scale per group of elements."""
    orig_shape = weight.shape
    flat = weight.reshape(-1)
    n_groups = (flat.numel() + group_size - 1) // group_size
    
    # Pad if needed
    padded = torch.zeros(n_groups * group_size)
    padded[:flat.numel()] = flat
    
    groups = padded.reshape(n_groups, group_size)
    scales = []
    q_groups = []
    for g in range(n_groups):
        gq, scale = symmetric_quantize(groups[g], n_bits)
        q_groups.append(gq)
        scales.append(scale)
    
    q_flat = torch.cat([g.flatten() for g in q_groups])[:flat.numel()]
    return q_flat.reshape(orig_shape), torch.tensor(scales)


# Compare granularities
# Simulate a weight matrix with varying scales per row
weight = torch.randn(256, 256)
# Make some rows have larger values (simulating outliers)
weight[0] *= 10
weight[5] *= 5
weight[100] *= 8

print(f"Weight matrix shape: {weight.shape}")
print(f"Weight range: [{weight.min():.2f}, {weight.max():.2f}]")
print()

for n_bits in [8, 4]:
    print(f"\n{'='*50}")
    print(f"INT{n_bits} Quantization:")
    
    # Per-tensor
    wq_pt, s_pt = per_tensor_quantize(weight, n_bits)
    w_hat_pt = wq_pt.float() * s_pt
    error_pt = (weight - w_hat_pt).abs().mean()
    
    # Per-channel
    wq_pc, s_pc = per_channel_quantize(weight, n_bits)
    w_hat_pc = wq_pc.float() * s_pc.unsqueeze(1)
    error_pc = (weight - w_hat_pc).abs().mean()
    
    # Per-group (g=128)
    wq_pg, s_pg = per_group_quantize(weight, n_bits, group_size=128)
    # Dequantize per-group (simplified)
    flat = weight.reshape(-1)
    q_flat = wq_pg.reshape(-1)
    n_groups = len(s_pg)
    w_hat_pg = torch.zeros_like(flat)
    for g in range(n_groups):
        start = g * 128
        end = min(start + 128, flat.numel())
        w_hat_pg[start:end] = q_flat[start:end].float() * s_pg[g]
    error_pg = (flat - w_hat_pg).abs().mean()
    
    print(f"  Per-tensor:        MAE = {error_pt:.6f}  (1 scale)")
    print(f"  Per-channel:       MAE = {error_pc:.6f}  ({weight.shape[0]} scales)")
    print(f"  Per-group (g=128): MAE = {error_pg:.6f}  ({n_groups} scales)")

---

## 4. PTQ vs QAT

### Post-Training Quantization (PTQ)
- Quantize AFTER training is complete
- No retraining needed
- Uses calibration data to find optimal scales
- Examples: GPTQ, AWQ, GGUF

### Quantization-Aware Training (QAT)
- Simulate quantization DURING training
- Model learns to be robust to quantization noise
- Better quality but requires training resources
- Uses Straight-Through Estimator (STE) for backprop through rounding

In [ ]:
# Demonstrate the Straight-Through Estimator concept

class FakeQuantize(torch.autograd.Function):
    """
    Fake quantization: quantize in forward, pass gradients straight through in backward.
    This is the core of QAT.
    """
    @staticmethod
    def forward(ctx, x, n_bits=8):
        qmax = 2 ** (n_bits - 1) - 1
        scale = x.abs().max() / qmax
        x_q = torch.round(x / scale) * scale  # quantize and immediately dequantize
        return x_q
    
    @staticmethod
    def backward(ctx, grad_output):
        # Straight-through: pass gradient unchanged
        return grad_output, None


# Demo: show that gradients flow through fake quantization
x = torch.randn(10, requires_grad=True)
y = FakeQuantize.apply(x, 4)  # INT4 fake quantization
loss = y.sum()
loss.backward()

print("Fake Quantization (STE) Demo:")
print(f"  Input x:              {x.data[:5].tolist()}")
print(f"  Fake-quantized y:     {y.data[:5].tolist()}")
print(f"  Gradient (dL/dx):     {x.grad[:5].tolist()}")
print(f"  Note: gradients are all 1.0 (straight-through)")

---

## 5. GPTQ: Post-Training Quantization for GPT Models

GPTQ (Frantar et al., 2022) uses the Optimal Brain Quantizer (OBQ) approach:

1. Quantize weights one column at a time
2. After quantizing each column, adjust remaining columns to compensate for the error
3. Use the Hessian (second-order information from calibration data) to find optimal adjustments

Key idea: When we quantize weight $w_i$, we introduce an error. We can partially compensate by adjusting the remaining unquantized weights $w_{i+1}, ..., w_n$.

In [ ]:
def simple_gptq_quantize(
    weight: torch.Tensor,  # (out_features, in_features)
    n_bits: int = 4,
    group_size: int = 128
) -> tuple:
    """
    Simplified GPTQ-style quantization.
    
    This is a simplified version that demonstrates the column-by-column
    approach without the full Hessian-based compensation.
    """
    W = weight.clone()
    out_features, in_features = W.shape
    
    qmax = 2 ** (n_bits - 1) - 1
    
    # Process column by column
    W_q = torch.zeros_like(W)
    scales = []
    
    for col in range(in_features):
        # Get current column
        w_col = W[:, col]
        
        # Quantize this column
        abs_max = w_col.abs().max()
        scale = abs_max / qmax if abs_max > 0 else 1.0
        w_col_q = torch.clamp(torch.round(w_col / scale), -qmax, qmax)
        
        # Store quantized values
        W_q[:, col] = w_col_q
        scales.append(scale)
        
        # Compute quantization error
        error = w_col - w_col_q * scale
        
        # Simple compensation: distribute error to next column
        if col < in_features - 1:
            W[:, col + 1] += error * 0.1  # simplified compensation
    
    return W_q, torch.tensor(scales)


# Compare simple quantization vs GPTQ-style
weight = torch.randn(512, 512)

# Naive per-tensor
wq_naive, s_naive = symmetric_quantize(weight.flatten(), n_bits=4)
w_hat_naive = wq_naive.reshape(weight.shape).float() * s_naive
error_naive = (weight - w_hat_naive).abs().mean()

# GPTQ-style
wq_gptq, s_gptq = simple_gptq_quantize(weight, n_bits=4)
w_hat_gptq = wq_gptq * s_gptq.unsqueeze(0)
error_gptq = (weight - w_hat_gptq).abs().mean()

print(f"INT4 Quantization Comparison (512x512 weight matrix):")
print(f"  Naive per-tensor:  MAE = {error_naive:.6f}")
print(f"  GPTQ-style:        MAE = {error_gptq:.6f}")
print(f"  Improvement:       {(error_naive - error_gptq) / error_naive * 100:.1f}%")

---

## 6. AWQ: Activation-Aware Weight Quantization

AWQ (Lin et al., 2023) observes that not all weights are equally important. Weights connected to large activations have more impact on output quality.

**Key idea**: Scale up important weights (those corresponding to salient activation channels) before quantization, so they get higher precision.

In [ ]:
def awq_quantize(
    weight: torch.Tensor,
    activation_magnitudes: torch.Tensor,  # average |activation| per input channel
    n_bits: int = 4,
    alpha: float = 0.5  # how much to use activation info
) -> tuple:
    """
    Simplified AWQ-style quantization.
    
    Scale important channels up before quantization.
    """
    # Compute per-channel scaling based on activation magnitudes
    # Channels with larger activations get scaled up (more precision)
    channel_importance = activation_magnitudes / activation_magnitudes.mean()
    scaling_factor = channel_importance.pow(alpha)
    
    # Scale weights
    W_scaled = weight * scaling_factor.unsqueeze(0)  # broadcast across output dim
    
    # Quantize scaled weights
    wq, scale = symmetric_quantize(W_scaled.flatten(), n_bits)
    wq = wq.reshape(weight.shape)
    
    # Dequantize and undo scaling
    w_hat = (wq.float() * scale) / scaling_factor.unsqueeze(0)
    
    return wq, scale, scaling_factor, w_hat


# Demo: AWQ vs naive quantization
weight = torch.randn(256, 256)

# Simulate activation magnitudes (some channels are much more active)
act_mags = torch.ones(256)
act_mags[10:15] = 10.0   # salient channels
act_mags[100:105] = 8.0  # salient channels

# Naive quantization
wq_naive, s_naive = symmetric_quantize(weight.flatten(), 4)
w_hat_naive = wq_naive.reshape(weight.shape).float() * s_naive

# AWQ quantization
wq_awq, s_awq, sf_awq, w_hat_awq = awq_quantize(weight, act_mags, n_bits=4)

# Compute error weighted by activation importance
weighted_error_naive = ((weight - w_hat_naive) * act_mags.unsqueeze(0)).abs().mean()
weighted_error_awq = ((weight - w_hat_awq) * act_mags.unsqueeze(0)).abs().mean()

print("AWQ vs Naive INT4 Quantization:")
print(f"  Naive weighted error:  {weighted_error_naive:.6f}")
print(f"  AWQ weighted error:    {weighted_error_awq:.6f}")
print(f"  AWQ improvement:       {(weighted_error_naive - weighted_error_awq)/weighted_error_naive*100:.1f}%")
print()
print("AWQ reduces error where it matters most (salient activation channels).")

---

## 7. INT8/INT4 Matrix Multiplication

How does quantized inference actually work? The key operation is:

$$Y = X \cdot W^T$$

With weight quantization ($W \approx s_W \cdot W_q$):

$$Y \approx X \cdot (s_W \cdot W_q)^T = s_W \cdot (X \cdot W_q^T)$$

The inner matmul $X \cdot W_q^T$ can use:
- FP16 x INT8 mixed-precision (requires dequantization per group)
- INT8 x INT8 (requires quantizing activations too)

In [ ]:
def quantized_matmul_demo(
    X: torch.Tensor,      # (batch, in_features) FP16 activations
    W: torch.Tensor,      # (out_features, in_features) FP32 weights
    n_bits: int = 8
):
    """
    Demonstrate weight-only quantized matrix multiplication.
    """
    # 1. Full precision reference
    Y_fp32 = torch.mm(X.float(), W.t())
    
    # 2. Quantize weights
    qmax = 2 ** (n_bits - 1) - 1
    # Per-channel quantization
    scales = W.abs().max(dim=1).values / qmax
    W_q = torch.clamp(torch.round(W / scales.unsqueeze(1)), -qmax, qmax)
    
    # 3. "Quantized" matmul: dequantize during computation
    # In practice, this is done in fused kernels (e.g., CUTLASS)
    W_dequant = W_q * scales.unsqueeze(1)
    Y_quant = torch.mm(X.float(), W_dequant.t())
    
    # 4. Compute error
    error = (Y_fp32 - Y_quant).abs().mean()
    relative_error = error / Y_fp32.abs().mean() * 100
    
    return {
        'y_fp32': Y_fp32,
        'y_quant': Y_quant,
        'mae': error.item(),
        'relative_error_pct': relative_error.item(),
        'weight_size_mb': W.numel() * n_bits / 8 / 1e6,
        'original_size_mb': W.numel() * 4 / 1e6,  # FP32
    }


# Test with a realistic-sized layer
X = torch.randn(1, 4096)  # batch=1, d_model=4096
W = torch.randn(4096, 4096)  # linear layer

print(f"Quantized MatMul Demo (4096x4096 weight matrix):")
print("=" * 60)

for n_bits in [8, 4, 3, 2]:
    result = quantized_matmul_demo(X, W, n_bits)
    print(f"  INT{n_bits}: MAE={result['mae']:.4f}, "
          f"Rel.Error={result['relative_error_pct']:.2f}%, "
          f"Size={result['weight_size_mb']:.1f}MB "
          f"({result['original_size_mb']/result['weight_size_mb']:.1f}x compression)")

---

## 8. GGUF Format Overview

GGUF (GPT-Generated Unified Format) is the format used by llama.cpp for running models on CPUs and various hardware.

Key features:
- Multiple quantization types: Q4_0, Q4_1, Q5_0, Q5_1, Q8_0, etc.
- Block-based quantization (typically 32 elements per block)
- Metadata stored in the file header
- Designed for CPU inference (but also used on GPU)

| Format | Bits/Weight | Block Size | Scale Storage | Quality |
|--------|------------|------------|---------------|--------|
| Q2_K   | 2.5625     | 256        | 6-bit scales  | Low    |
| Q3_K_M | 3.4375     | 256        | 6-bit scales  | Medium |
| Q4_K_M | 4.8125     | 256        | 6-bit scales  | Good   |
| Q5_K_M | 5.6875     | 256        | 6-bit scales  | Very Good |
| Q6_K   | 6.5625     | 256        | 8-bit scales  | Excellent |
| Q8_0   | 8.5        | 32         | FP16 scale    | Near-FP16 |

In [ ]:
# Quality vs Size trade-off visualization

# Approximate quality (perplexity increase) and sizes for a 7B model
quant_methods = {
    'FP16': {'bits': 16, 'size_gb': 14.0, 'ppl_increase': 0.0, 'color': '#3498db'},
    'INT8 (Q8_0)': {'bits': 8.5, 'size_gb': 7.5, 'ppl_increase': 0.01, 'color': '#2ecc71'},
    'GPTQ INT4': {'bits': 4.5, 'size_gb': 4.0, 'ppl_increase': 0.1, 'color': '#f39c12'},
    'AWQ INT4': {'bits': 4.5, 'size_gb': 4.0, 'ppl_increase': 0.08, 'color': '#e67e22'},
    'Q4_K_M': {'bits': 4.8, 'size_gb': 4.3, 'ppl_increase': 0.05, 'color': '#e74c3c'},
    'Q3_K_M': {'bits': 3.4, 'size_gb': 3.1, 'ppl_increase': 0.3, 'color': '#9b59b6'},
    'Q2_K': {'bits': 2.6, 'size_gb': 2.4, 'ppl_increase': 1.5, 'color': '#1abc9c'},
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Size vs Quality
for name, info in quant_methods.items():
    axes[0].scatter(info['size_gb'], info['ppl_increase'], 
                    s=200, c=info['color'], zorder=5, edgecolors='black', linewidths=1.5)
    axes[0].annotate(name, xy=(info['size_gb'], info['ppl_increase']),
                     xytext=(info['size_gb']+0.3, info['ppl_increase']+0.05),
                     fontsize=9)

axes[0].set_xlabel('Model Size (GB)', fontsize=12)
axes[0].set_ylabel('Perplexity Increase (lower is better)', fontsize=12)
axes[0].set_title('Quality vs Size Trade-off (7B Model)', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Throughput improvement
names = list(quant_methods.keys())
fp16_size = quant_methods['FP16']['size_gb']
speedups = [fp16_size / info['size_gb'] for info in quant_methods.values()]
colors = [info['color'] for info in quant_methods.values()]

bars = axes[1].barh(names, speedups, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Theoretical Decode Speedup vs FP16', fontsize=12)
axes[1].set_title('Throughput Improvement (Memory-Bound)', fontsize=14)
for bar, s in zip(bars, speedups):
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{s:.1f}x', va='center', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercises

### Exercise 1: Implement Quantization/Dequantization

Implement per-group asymmetric quantization and verify it with a round-trip test.

In [ ]:
def per_group_asymmetric_quantize(
    weight: torch.Tensor,
    n_bits: int = 4,
    group_size: int = 128
) -> tuple:
    """
    Exercise: Implement per-group asymmetric quantization.
    
    For each group of `group_size` elements:
    1. Find min and max
    2. Compute scale and zero_point
    3. Quantize to [0, 2^n_bits - 1]
    
    Return: (quantized_weights, scales, zero_points)
    """
    # YOUR CODE HERE
    pass

# Test:
# w = torch.randn(256, 256)
# wq, scales, zps = per_group_asymmetric_quantize(w, n_bits=4, group_size=128)
# w_hat = ...  # dequantize
# error = (w - w_hat).abs().mean()
# print(f"MAE: {error:.6f}")

### Exercise 2: Compare Quantization Methods

Create a comprehensive benchmark comparing different quantization methods on a simulated neural network layer.

In [ ]:
def benchmark_quantization(
    weight: torch.Tensor,
    input_data: torch.Tensor,
    n_bits: int = 4
):
    """
    Exercise: Compare quantization methods by measuring:
    1. Weight MAE
    2. Output MAE (how much the layer output changes)
    3. Output cosine similarity (direction preservation)
    
    Test methods:
    - Per-tensor symmetric
    - Per-channel symmetric  
    - Per-group symmetric (g=128)
    - Per-group symmetric (g=32)
    """
    # YOUR CODE HERE
    pass

# Test:
# W = torch.randn(4096, 4096)  # linear layer
# X = torch.randn(32, 4096)    # batch of inputs
# benchmark_quantization(W, X, n_bits=4)

---

## Summary

### Key Takeaways

1. **Number formats** trade precision for memory: FP32 (4B) -> FP16/BF16 (2B) -> INT8 (1B) -> INT4 (0.5B). BF16 has the same range as FP32 but less precision.

2. **Symmetric quantization** maps zero to zero and uses one scale. Asymmetric adds a zero-point for non-centered distributions.

3. **Granularity** matters: per-group quantization provides much better accuracy than per-tensor, with only a small overhead for storing scales.

4. **GPTQ** quantizes column-by-column with error compensation using Hessian information. Good for INT4.

5. **AWQ** preserves quality by protecting weights connected to salient activation channels. Often slightly better than GPTQ.

6. **For memory-bound decode**: quantization directly translates to throughput improvement (2x for INT8, 4x for INT4) because we read fewer bytes from HBM.

7. **Quality trade-off**: INT8 is nearly lossless. INT4 with good methods (GPTQ/AWQ) has minimal quality loss. Below INT4, quality degrades significantly.

### What's Next

In **Chapter 1.8: Sampling Strategies**, we'll explore how to convert model logits into text, covering greedy decoding, temperature, top-k, top-p, and other sampling methods.